In [90]:
import pandas as pd

df = pd.read_csv("data/cs-training.csv", index_col=0)
print("Размер датасета:", df.shape)
df.head()

Размер датасета: (150000, 11)


,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [91]:
print("Число пропусков:")
print(df.isna().sum())
print()
print(df["SeriousDlqin2yrs"].value_counts(normalize=True))

Число пропусков:
SeriousDlqin2yrs                            0
RevolvingUtilizationOfUnsecuredLines        0
age                                         0
NumberOfTime30-59DaysPastDueNotWorse        0
DebtRatio                                   0
MonthlyIncome                           29731
NumberOfOpenCreditLinesAndLoans             0
NumberOfTimes90DaysLate                     0
NumberRealEstateLoansOrLines                0
NumberOfTime60-89DaysPastDueNotWorse        0
NumberOfDependents                       3924
dtype: int64

SeriousDlqin2yrs
0    0.93316
1    0.06684
Name: proportion, dtype: float64


In [92]:
mInc = df["MonthlyIncome"].median()
mDep = df["NumberOfDependents"].median()

In [93]:
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(mInc)
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(mDep)

In [94]:
X = df.drop(columns=["SeriousDlqin2yrs"])
X.head()

,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
2,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
3,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
4,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
5,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [95]:
y = df["SeriousDlqin2yrs"]
y.head()

1    1
2    0
3    0
4    0
5    0
Name: SeriousDlqin2yrs, dtype: int64

In [96]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test= train_test_split(X, y, test_size=0.2, random_state=67, stratify=y)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print()
print(y_train.value_counts(normalize=True), y_test.value_counts(normalize=True))

(120000, 10) (30000, 10) (120000,) (30000,)

SeriousDlqin2yrs
0    0.933158
1    0.066842
Name: proportion, dtype: float64 SeriousDlqin2yrs
0    0.933167
1    0.066833
Name: proportion, dtype: float64


In [97]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.set_output(transform="pandas")
X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)


In [98]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)

model.fit(X_train_scaled, y_train)

probabilities = model.predict_proba(X_test_scaled)[:,1]


In [99]:
from sklearn.metrics import roc_auc_score

auc = roc_auc_score(y_test, probabilities)

gini = 2*auc - 1

print("AUC:", auc, "   GINI:", gini*100)

AUC: 0.6950308369102249    GINI: 39.006167382044985
